In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from data.datasets import generate_x_y_data_v1

X, Y = generate_x_y_data_v1(isTrain=True, batch_size=32)
print(X.shape, Y.shape)  # (10, 32, 2) (10, 32, 2)

In [ ]:
from data.datasets import generate_x_y_data_v1

X_train, Y_train = generate_x_y_data_v1(isTrain=True, batch_size=5000)
X_val, Y_val = generate_x_y_data_v1(isTrain=False, batch_size=1000)

X_train = X_train.transpose(1, 0, 2) 
Y_train = Y_train.transpose(1, 0, 2)
X_val = X_val.transpose(1, 0, 2)
Y_val = Y_val.transpose(1, 0, 2)

print(X_train.shape, Y_train.shape)  

In [ ]:
from tensorflow import keras

rnn = keras.Sequential([
    keras.layers.Input(shape=(10, 2)),
    keras.layers.SimpleRNN(64),        # son gizli durum: (64,)
    keras.layers.Dense(10 * 2),
    keras.layers.Reshape((10, 2)),
])

rnn.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse")
rnn.summary()

In [ ]:
early = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

history = rnn.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=100,
    batch_size=64,
    callbacks=[early],
    verbose=1,
)

print("En iyi val_loss:", min(history.history["val_loss"]))

In [ ]:
i = 0
pred = rnn.predict(X_val[i:i+1], verbose=0)[0]

plt.figure(figsize=(10, 3))
plt.plot(range(10), X_val[i, :, 0], 'o--', label='girdi (sin)')
plt.plot(range(10, 20), Y_val[i, :, 0], 'x--', label='gerçek (sin)')
plt.plot(range(10, 20), pred[:, 0], 'o--', label='tahmin (sin)')
plt.legend()
plt.title('SimpleRNN: tahmin vs gerçek')
plt.show()

In [ ]:
rnn.save("../model/rnn_sine_wave.keras")